# Wrangling Harmonic-EEG Data

This notebook gets the Harmonic-EEG dataset ready for analysis. The dataset has EEG recordings from 40 musicians, either classical musicians or jazz-trained improvisers, while they listened to three-chord progressions where the middle chord was occasionally changed. Before analyzing the EEG, we check who the participants are and that every participant's files are present and complete.

<details>
<summary><b>What's in <code>dataset_description.json</code></b> (click to expand)</summary>

`dataset_description.json` is the dataset's title page, required by the BIDS standard. It holds information about the dataset as a whole, not about any participant:
- the dataset's name ("EEG: Improvisation and Musical Structures"), version (1.0.0) and the BIDS version it follows (1.1.1)
- the authors, funding, ethics approval and acknowledgements
- the papers to cite (Goldman et al., 2020) and the license (CC0, free to use)
- its DOI on NEMAR, and that it was copied from OpenNeuro dataset `ds003570`

</details>

<details>
<summary><b>Hidden files in the <code>Data</code> folder</b> (click to expand)</summary>

Besides the data itself, the downloaded `Data` folder has a few hidden files and folders (names starting with `.`) used to manage and publish the dataset. None of them hold data we analyze:
- `.datalad/`: marks `Data` as a DataLad dataset (a tool for versioning and downloading large datasets) and stores its ID
- `.gitattributes`: tells git-annex (used by DataLad) which files to store as large files; here the big `.set` and `.fdt` EEG files are stored as links to the actual data
- `.github/`: automated checks the dataset's publishers run on GitHub, e.g. confirming the dataset follows the BIDS standard
- `.nemar/`: information for NEMAR, the website that hosts the dataset: a description of the study and a report confirming all 80 EEG files are present (it checks that the files exist, not that each recording is complete)
- `.bidsignore`: tells the BIDS validator to skip the `.nemar/` folder, since it isn't part of the BIDS standard

</details>

### Import libraries

Load the Python libraries used in this notebook: `pandas` for working with tables, `pathlib` for finding files in the `Data` folder, `json` for reading the dataset's `.json` metadata files, `numpy` for comparing lists of numbers, and `scipy.io` for reading the MATLAB-format `.set` files.

In [40]:
import pandas as pd
from pathlib import Path
import json
import numpy as np
import scipy.io

### Load and summarize participant information

`participants.tsv` has one row per participant with their ID, group (classical or improviser), handedness, age and gender. The cells below load it and summarize who is in the study:
- how many participants there are in total and in each group
- the average age and age range
- how many of each gender and handedness are in each group

In [41]:
# Load the TSV file into a DataFrame
participants_data = pd.read_csv('Data/participants.tsv', sep='\t')

# Convert to dataframe
df = pd.DataFrame(participants_data)

# Display the first few rows
df.head()

,participant_id,group,handedness,age,gender
0,sub-001,Cl,R,26,M
1,sub-002,Cl,R,26,M
2,sub-003,Cl,R,42,F
3,sub-004,Cl,R,24,F
4,sub-005,Imp,L,27,M


In [42]:
# Full names for the group codes used in the TSV
group_names = {"Cl": "classical musicians who don't improvise",
               "Imp": "improvisers (jazz-trained)"}

# Number of unique participants (participant_id is the primary key)
print(f"Participants: {df['participant_id'].nunique()}")

# Number of participants in each group
print("Group:")
for g, n in df["group"].value_counts().items():
    print(f"  {g} ({group_names[g]}): {n}")

# Mean and range of age
print(f"Age: mean {df['age'].mean():.1f}, range {df['age'].min()}–{df['age'].max()}")

# Number of participants of each gender
print("Gender:", df["gender"].value_counts().to_dict())

# Number of participants of each handedness
print("Handedness:", df["handedness"].value_counts().to_dict())

Participants: 40
Group:
  Imp (improvisers (jazz-trained)): 24
  Cl (classical musicians who don't improvise): 16
Age: mean 25.3, range 18–42
Gender: {'M': 24, 'F': 16}
Handedness: {'R': 31, 'A': 5, 'L': 4}


In [43]:
# Summary statistics for age
df["age"].agg(["count", "mean", "std", "min", "max"])

count    40.000000
mean     25.275000
std       5.514491
min      18.000000
max      42.000000
Name: age, dtype: float64

In [44]:
# Number of participants of each gender in each group
display(pd.crosstab(df["group"], df["gender"], margins=True))

# Number of participants of each handedness in each group
display(pd.crosstab(df["group"], df["handedness"], margins=True))

gender,F,M,All
group,,,
Cl,12,4,16
Imp,4,20,24
All,16,24,40


handedness,A,L,R,All
group,,,,
Cl,1,1,14,16
Imp,4,3,17,24
All,5,4,31,40


### Check that every participant has all their files

Each participant's `eeg` folder should contain the same set of files: the EEG recording itself (split into a `.set` header and an `.fdt` data file) plus smaller files describing the channels, electrodes and events. This cell lists every file with its size, then reports:
- any participant who is missing a file
- any EEG recording under 1 MB, which usually means it was never fully downloaded

In [45]:
root = Path("Data")

# List every file in each participant's eeg folder
rows = []
for file in sorted(root.glob("sub-*/eeg/*")):
    rows.append({
        # "sub-001_task-AuditoryOddballChords_events.tsv" -> "sub-001"
        "participant": file.name.split("_")[0],
        # "sub-001_task-AuditoryOddballChords_events.tsv" -> "events.tsv"
        "file": file.name.split("_")[-1],
        # Size in MB (0 if the file is a link to data that hasn't been downloaded)
        "size_MB": round(file.stat().st_size / 1e6, 2) if file.exists() else 0,
    })
files = pd.DataFrame(rows)

# Table with one row per participant and one column per file type (NaN = missing)
sizes = files.pivot(index="participant", columns="file", values="size_MB")

# Number of participants found
print(len(sizes), "participants found")

# Number of participants that have each file type (should all match the number above)
print("\nFiles per type:")
print(sizes.notna().sum())

# Participants missing at least one file type
missing = sizes[sizes.isna().any(axis=1)]
print("\nParticipants with missing files:")
print(missing if len(missing) else "  none")

# EEG recordings under 1 MB weren't fully downloaded
small = sizes.loc[(sizes["eeg.set"] < 1) | (sizes["eeg.fdt"] < 1), ["eeg.set", "eeg.fdt"]]
print("\nEEG recordings under 1 MB:")
print(small if len(small) else "  none")

40 participants found

Files per type:
file
channels.tsv        40
coordsystem.json    40
eeg.fdt             40
eeg.json            40
eeg.set             40
electrodes.tsv      40
events.json         40
events.tsv          40
dtype: int64

Participants with missing files:
  none

EEG recordings under 1 MB:
  none


### Check each participant's channels.tsv

`channels.tsv` lists the EEG channels (electrodes) in a recording. Under the BIDS standard it can have each channel's `name`, `type` (e.g. EEG or EOG), `units` (e.g. µV) and `status` (good or bad). This cell reads every participant's file and makes one row per participant showing:
- how many channels they have (should be 64)
- which columns their file has
- how many `type` and `units` values are missing (`n/a`)
- whether any channel name appears twice
- whether their channel list (names and order) matches everyone else's

In [32]:
# Read every participant's channels.tsv ("n/a" is read as missing)
channels = {f.name.split("_")[0]: pd.read_csv(f, sep="\t")
            for f in sorted(root.glob("sub-*/eeg/*_channels.tsv"))}

# The most common channel list (names in order), to compare each participant against
usual_names = pd.Series([tuple(c["name"]) for c in channels.values()]).mode()[0]

# One row per participant summarizing their channels.tsv
rows = []
for participant, c in channels.items():
    rows.append({
        "participant": participant,
        "n_channels": len(c),
        "columns": ", ".join(c.columns),
        "missing_type": c["type"].isna().sum(),
        "missing_units": c["units"].isna().sum(),
        "duplicate_names": c["name"].duplicated().sum(),
        "same_channels_as_others": tuple(c["name"]) == usual_names,
    })
channel_report = pd.DataFrame(rows).set_index("participant")

# How many participants share each combination of results (dropna=False keeps missing values)
channel_summary = channel_report.value_counts(dropna=False).to_frame("participants")
display(channel_summary)

# Full report only if participants' files differ
if len(channel_summary) > 1:
    display(channel_report)

,,,,,,participants
n_channels,columns,missing_type,missing_units,duplicate_names,same_channels_as_others,
64,"name, type, units",64,64,0,True,40


### Check each participant's coordsystem.json

`coordsystem.json` describes how to read the electrode positions in `electrodes.tsv`: which coordinate system they use and in what units. Under the BIDS standard it must have:
- `EEGCoordinateSystem`: the name of the coordinate system, or `Other` if it isn't a standard one
- `EEGCoordinateUnits`: the units of the positions (`m`, `mm`, `cm` or `n/a`)
- `EEGCoordinateSystemDescription`: a description of the system, required when it is `Other`

This cell reads every participant's file into one row per participant with one column per field (NaN = field missing). If every participant's file is the same, only a one-line summary is shown.

In [33]:
# Read every participant's coordsystem.json (one row per participant, one column per field)
rows = []
for f in sorted(root.glob("sub-*/eeg/*_coordsystem.json")):
    with open(f) as file:
        row = json.load(file)
    row["participant"] = f.name.split("_")[0]
    rows.append(row)
coord_report = pd.DataFrame(rows).set_index("participant")

# How many participants share each combination of values (dropna=False keeps missing fields)
coord_summary = coord_report.value_counts(dropna=False).to_frame("participants")
display(coord_summary)

# Full report only if participants' files differ
if len(coord_summary) > 1:
    display(coord_report)

,,,participants
EEGCoordinateUnits,EEGCoordinateSystem,EEGCoordinateSystemDescription,
mm,Other,EEGLAB,40


### Check each participant's electrodes.tsv

`electrodes.tsv` gives the position of each electrode on the head: its `name` and its `x`, `y`, `z` coordinates, in the coordinate system and units described in `coordsystem.json`. This cell reads every participant's file and makes one row per participant showing:
- how many electrodes they have (should be 64)
- which columns their file has
- how many values are missing (blank or `n/a`)
- whether their electrode names (in order) match everyone else's
- whether their electrode positions (x, y, z for each electrode, in order) match everyone else's

If every participant's results are the same, only a one-line summary is shown.

In [46]:
# Read every participant's electrodes.tsv ("n/a" and blanks are read as missing)
electrodes = {f.name.split("_")[0]: pd.read_csv(f, sep="\t")
              for f in sorted(root.glob("sub-*/eeg/*_electrodes.tsv"))}

# The most common electrode name list (in order), to compare each participant against
usual_electrodes = pd.Series([tuple(e["name"]) for e in electrodes.values()]).mode()[0]

# The most common set of positions (x, y, z for each electrode, in order), to compare against
usual_positions = pd.Series([tuple(e[["x", "y", "z"]].itertuples(index=False))
                             for e in electrodes.values()]).mode()[0]

# One row per participant summarizing their electrodes.tsv
rows = []
for participant, e in electrodes.items():
    rows.append({
        "participant": participant,
        "n_electrodes": len(e),
        "columns": ", ".join(e.columns),
        "missing_values": e.isna().sum().sum(),
        "same_names_as_others": tuple(e["name"]) == usual_electrodes,
        "same_positions_as_others": tuple(e[["x", "y", "z"]].itertuples(index=False)) == usual_positions,
    })
electrode_report = pd.DataFrame(rows).set_index("participant")

# How many participants share each combination of results (dropna=False keeps missing values)
electrode_summary = electrode_report.value_counts(dropna=False).to_frame("participants")
display(electrode_summary)

# Full report only if participants' files differ
if len(electrode_summary) > 1:
    display(electrode_report)

,,,,,participants
n_electrodes,columns,missing_values,same_names_as_others,same_positions_as_others,
64,"name, x, y, z",0,True,True,40


### Check each participant's events.json

`events.json` is meant to explain the columns of `events.tsv`: what each column means and what each event code stands for (for example, that `fourRoot_S` is a standard trial whose middle chord is IV in root position). Under the BIDS standard this file is optional, and when present it should be a JSON object (`{...}`).

In this dataset every participant's file is an empty list (`[]`). So, the meaning of the event codes has to come from the dataset README and the paper instead.

This cell reads every participant's file and makes one row per participant showing what kind of JSON it holds (BIDS expects `dict`) and how many entries it has. If every participant's results are the same, only a one-line summary is shown.

In [48]:
# Read every participant's events.json and record what it holds
rows = []
for f in sorted(root.glob("sub-*/eeg/*_events.json")):
    with open(f) as file:
        contents = json.load(file)
    rows.append({
        "participant": f.name.split("_")[0],
        # "dict" for a JSON object {...}, "list" for a JSON list [...]
        "json_type": type(contents).__name__,
        "n_entries": len(contents),
    })
events_json_report = pd.DataFrame(rows).set_index("participant")

# How many participants share each combination of results (dropna=False keeps missing values)
events_json_summary = events_json_report.value_counts(dropna=False).to_frame("participants")
display(events_json_summary)

# Full report only if participants' files differ
if len(events_json_summary) > 1:
    display(events_json_report)

,,participants
json_type,n_entries,
list,0,40


### Check each participant's events.tsv

`events.tsv` lists everything that happened during the recording, one row per event: when it happened (`onset`, in seconds from the start of the recording), how long it lasted (`duration`) and what it was (`value`). In this dataset all the information is in the `value` column:
- each trial is three chords 400 ms apart: `two` (the ii chord), a middle chord, then `one` (the I chord)
- the middle chord's code says which chord it was and what kind of trial it was, e.g. `fourRoot_S` = IV in root position on a standard trial, `five6_deviantE...` = an exemplar deviant, `four6_deviantF...` = a function deviant
- codes like `Correct - Function!` and the `correct`/`incorrect` endings appear to record the participant's responses (to confirm against the paper)
- `16128` marks the start of a block, and `boundary` marks where EEGLAB joined two blocks of recording together

This cell reads every participant's file and makes one row per participant showing the number of events, which columns are completely empty, missing values in the other columns, whether the events are in time order, the time of the last event, the number of different codes, and how many standard, exemplar deviant and function deviant trials there are. Checks where every participant has the same result are shown once; checks where they differ are shown for every participant. A last table lists every event code, how many participants have it, and the fewest and most times it appears for one participant.

In [49]:
# Read every participant's events.tsv ("n/a" and blanks are read as missing)
events = {f.name.split("_")[0]: pd.read_csv(f, sep="\t")
          for f in sorted(root.glob("sub-*/eeg/*_events.tsv"))}

# One row per participant summarizing their events.tsv
rows = []
for participant, e in events.items():
    rows.append({
        "participant": participant,
        "n_events": len(e),
        "columns": ", ".join(e.columns),
        # Columns with no values at all
        "empty_columns": ", ".join(e.columns[e.isna().all()]),
        # Number of missing values in each of the other columns, e.g. "missing_duration"
        **e.loc[:, e.notna().any()].isna().sum().add_prefix("missing_"),
        "onsets_in_order": e["onset"].is_monotonic_increasing,
        "last_onset_s": round(e["onset"].max()),
        "n_codes": e["value"].nunique(),
        # Middle-chord codes: "_S" = standard, "deviantE" = exemplar, "deviantF" = function
        "n_standard": e["value"].str.contains("_S").sum(),
        "n_exemplar_deviant": e["value"].str.contains("deviantE").sum(),
        "n_function_deviant": e["value"].str.contains("deviantF").sum(),
        "n_boundaries": (e["value"] == "boundary").sum(),
    })
events_report = pd.DataFrame(rows).set_index("participant")

# Checks where every participant has the same result, shown once
same_for_all = events_report.nunique(dropna=False) == 1
display(events_report.loc[:, same_for_all].iloc[0].to_frame("all participants"))

# Checks where participants differ, shown for every participant
display(events_report.loc[:, ~same_for_all])

# Every event code: how many participants have it, and the fewest / most times it appears
all_events = pd.concat(events, names=["participant", "row"]).reset_index()
code_counts = pd.crosstab(all_events["value"], all_events["participant"])
display(pd.DataFrame({"participants_with_code": (code_counts > 0).sum(axis=1),
                      "min_count": code_counts.min(axis=1),
                      "max_count": code_counts.max(axis=1)}))

,all participants
columns,"onset, duration, sample, trial_type, response_..."
empty_columns,"sample, response_time, stim_file"
missing_onset,0
missing_duration,7
missing_trial_type,0
missing_value,0
onsets_in_order,True
n_standard,1220
n_exemplar_deviant,109
n_function_deviant,111


,n_events,last_onset_s,n_codes
participant,,,
sub-001,4553,2350,23
sub-002,4583,2396,30
sub-003,4521,2343,30
sub-004,4566,2385,28
sub-005,4501,2445,28
sub-006,4553,2348,27
sub-007,4552,2337,26
sub-008,4732,2340,31
sub-009,4530,2347,29


,participants_with_code,min_count,max_count
value,,,
16128,39,0,8
16228,1,0,1
16368,2,0,1
Correct - Exemplar!,40,5,108
Correct - Function!,40,44,110
Incorrect - Standard!,39,0,270
boundary,40,7,7
five6_S,40,244,305
five6_Sincorrect,30,0,61


### Check each participant's eeg.json

`eeg.json` describes how a participant's EEG was recorded: the equipment (`ManufacturersModelName`), how many samples per second were taken (`SamplingFrequency`), how many channels of each kind there are (`EEGChannelCount`, `EOGChannelCount`, ...), the electrical reference, the mains power frequency, and how long the recording session lasted (`RecordingDuration`, in seconds).

This cell reads every participant's file into one row per participant with one column per field. Fields that are the same for everyone are shown once; fields that differ are shown for every participant.

In [50]:
# Read every participant's eeg.json (one row per participant, one column per field)
rows = []
for f in sorted(root.glob("sub-*/eeg/*_eeg.json")):
    with open(f) as file:
        row = json.load(file)
    row["participant"] = f.name.split("_")[0]
    rows.append(row)
eeg_json_report = pd.DataFrame(rows).set_index("participant")

# Number of fields missing across all files (a field one file has but another doesn't)
print("Missing fields:", eeg_json_report.isna().sum().sum())

# Fields where every participant has the same value, shown once
same_for_all = eeg_json_report.nunique(dropna=False) == 1
display(eeg_json_report.loc[:, same_for_all].iloc[0].to_frame("all participants"))

# Fields where participants differ, shown for every participant
display(eeg_json_report.loc[:, ~same_for_all])

Missing fields: 0


,all participants
InstitutionAddress,"550W 120th Street, New York, NY 10027, USA"
InstitutionName,"Columbia University, NY, USA"
InstitutionalDepartmentName,Laboratory for Intelligent Imaging and Neural ...
PowerLineFrequency,60
ManufacturersModelName,Biosemi - ActiveTwo
EEGChannelCount,64
TaskName,AuditoryOddballChords
MiscChannelCount,64
EEGReference,common
RecordingType,continuous


,RecordingDuration
participant,
sub-001,2354
sub-002,2402
sub-003,2347
sub-004,2390
sub-005,2449
sub-006,2350
sub-007,2339
sub-008,2341
sub-009,2351


### Check each participant's .set file

The EEG recording is stored in two files. The `.set` file is the header: a MATLAB file written by EEGLAB that describes the recording (number of channels, sampling rate, number of samples, channel names and positions, the event list) and names the `.fdt` file that holds the actual voltage values. The `.fdt` file can't be read without it, so it's checked first.

This cell opens every participant's `.set` file and lists any that can't be opened. For the rest, it makes one row per participant with the header values and checks that they agree with the participant's other files:
- the data file it points to is the participant's own `.fdt` file
- the channel names match `channels.tsv`
- the duration (samples ÷ sampling rate) matches `RecordingDuration` in `eeg.json`
- the event codes match the `value` column of `events.tsv`
- the event times match the `onset` column of `events.tsv`
- the electrode positions match `electrodes.tsv`
- no cleaning has been applied yet (no processing steps in the file's history, and no ICA)

It also shows the date each participant was recorded.

Checks where every participant has the same result are shown once; checks where they differ are shown for every participant.

In [51]:
# Read every participant's .set file (the header for their EEG recording)
rows = []
unreadable = []
for f in sorted(root.glob("sub-*/eeg/*_eeg.set")):
    participant = f.name.split("_")[0]
    try:
        EEG = scipy.io.loadmat(f, squeeze_me=True, struct_as_record=False)["EEG"]
    except OSError:
        # The file is cut short and can't be opened
        unreadable.append(participant)
        continue
    # Recording date, stored as [year, month, day, hour, minute, second]
    year, month, day = EEG.etc.T0[:3]
    rows.append({
        "participant": participant,
        "recording_date": f"{year}-{month:02d}-{day:02d}",
        "n_channels": EEG.nbchan,
        "sampling_rate": EEG.srate,
        "n_samples": EEG.pnts,
        "duration_s": EEG.pnts / EEG.srate,
        # Compare with the participant's other files
        "points_to_own_fdt": EEG.datfile == f.name.replace(".set", ".fdt"),
        "channels_match_channels_tsv": [c.labels for c in EEG.chanlocs] == list(channels[participant]["name"]),
        "duration_matches_eeg_json": EEG.pnts / EEG.srate == eeg_json_report.loc[participant, "RecordingDuration"],
        "events_match_events_tsv": [e.type for e in EEG.event] == list(events[participant]["value"]),
        # Event times are stored in samples, counting from 1
        "event_times_match_events_tsv": np.allclose([(e.latency - 1) / EEG.srate for e in EEG.event],
                                                    events[participant]["onset"]),
        # electrodes.tsv rounds positions to 2 decimals, so allow differences up to 0.005
        "positions_match_electrodes_tsv": np.allclose([[c.X, c.Y, c.Z] for c in EEG.chanlocs],
                                                      electrodes[participant][["x", "y", "z"]], atol=0.005),
        # No cleaning done: only EEGLAB's routine format check in the history, and no ICA
        "no_preprocessing": EEG.history.strip() == "EEG = eeg_checkset( EEG );" and np.size(EEG.icaweights) == 0,
    })
set_report = pd.DataFrame(rows).set_index("participant")

# Participants whose .set file can't be opened
print("Unreadable .set files:", unreadable)

# Checks where every participant has the same result, shown once
same_for_all = set_report.nunique() == 1
display(set_report.loc[:, same_for_all].iloc[0].to_frame("all readable files"))

# Checks where participants differ, shown for every participant
display(set_report.loc[:, ~same_for_all])

Unreadable .set files: ['sub-003', 'sub-004', 'sub-005', 'sub-013', 'sub-019', 'sub-022']


,all readable files
n_channels,64
sampling_rate,2048
points_to_own_fdt,True
channels_match_channels_tsv,True
duration_matches_eeg_json,True
events_match_events_tsv,True
event_times_match_events_tsv,True
positions_match_electrodes_tsv,True
no_preprocessing,True


,recording_date,n_samples,duration_s
participant,,,
sub-001,2017-02-07,4820992,2354.0
sub-002,2017-02-09,4919296,2402.0
sub-006,2017-03-02,4812800,2350.0
sub-007,2017-03-02,4790272,2339.0
sub-008,2017-03-03,4794368,2341.0
sub-009,2017-03-07,4814848,2351.0
sub-010,2017-03-07,4825088,2356.0
sub-011,2017-03-08,4808704,2348.0
sub-012,2017-03-08,4788224,2338.0


### Look inside a .set file

This shows everything stored in one participant's `.set` file (change `participant` to look at another; every readable file has the same fields). The tables are:
- **every field**: one row per field with its type, size (0 = empty) and a short preview
- **`chanlocs`**: the channel list, one row per channel, with its name (`labels`) and position (`X`, `Y`, `Z` and spherical coordinates)
- **`chaninfo`**: where the channel positions came from (`biosemi64.sph`, Biosemi's standard template layout, not measured on each participant) and which way the nose points (`+X`)
- **`event`**: the event list, one row per event, with its code (`type`), time in samples (`latency`, counting from 1) and its position in the original list (`urevent`)
- **`urevent`**: the original event list as recorded, before it was edited. Here each button press is the code `16224`; in `event` those were replaced by scored responses (e.g. `Correct - Function!`), and the middle chord was relabelled to say whether the response was correct. Its times (`latency`) restart at the beginning of each of the 8 blocks, so use `event` for timing
- **`reject`** and **`stats`**: where EEGLAB stores rejected data and data statistics; these are empty apart from EEGLAB's default display settings, as no cleaning has been done
- **`etc`**: the recording date and time (`T0`)

In [54]:
# Pick a participant to look inside their .set file
participant = "sub-001"
EEG = scipy.io.loadmat(next(root.glob(f"{participant}/eeg/*_eeg.set")),
                       squeeze_me=True, struct_as_record=False)["EEG"]

def preview(value):
    # A group of named sub-fields: list the sub-field names
    if isinstance(value, scipy.io.matlab.mat_struct):
        return "sub-fields: " + ", ".join(value._fieldnames)
    # A list of groups (e.g. one per event): list the sub-field names of the first one
    if isinstance(value, np.ndarray) and value.size and isinstance(value.flat[0], scipy.io.matlab.mat_struct):
        return "each entry has: " + ", ".join(value.flat[0]._fieldnames)
    return str(value)[:100]

# One row per field in the .set file
rows = []
for field in EEG._fieldnames:
    value = getattr(EEG, field)
    rows.append({"field": field, "type": type(value).__name__,
                 "size": np.size(value), "preview": preview(value)})
display(pd.DataFrame(rows).set_index("field"))

# Every field with more inside it, in the same order as the table above
for field in EEG._fieldnames:
    value = getattr(EEG, field)
    # A group of sub-fields: one row per sub-field
    if isinstance(value, scipy.io.matlab.mat_struct):
        print(field)
        display(pd.DataFrame([{"sub-field": k, "size": np.size(getattr(value, k)),
                               "preview": preview(getattr(value, k))}
                              for k in value._fieldnames]).set_index("sub-field"))
    # A list of groups (e.g. one per event): one row per entry, one column per sub-field
    elif isinstance(value, np.ndarray) and value.size and isinstance(value.flat[0], scipy.io.matlab.mat_struct):
        print(field)
        display(pd.DataFrame([{k: getattr(entry, k) for k in entry._fieldnames} for entry in value]))

,type,size,preview
field,,,
setname,str,1,context
filename,str,1,sub-001_task-AuditoryOddballChords_eeg.set
filepath,str,1,/media/datadrive/josef-home/_DATA_LIINC/MUSIC_...
subject,ndarray,0,[]
group,ndarray,0,[]
condition,ndarray,0,[]
session,ndarray,0,[]
comments,str,1,Original file: /Volumes/GOLDMAN BACKUP/MMN Fun...
nbchan,int,1,64


chanlocs


,sph_theta,sph_phi,labels,theta,radius,sph_radius,X,Y,Z,type,ref,urchan
0,18,-2,Fp1,-18.0,0.511111,1,0.950477,0.308829,-0.034899,[],[],1
1,36,-2,AF7,-36.0,0.511111,1,0.808524,0.587427,-0.034899,[],[],2
2,25,16,AF3,-25.0,0.411111,1,0.871199,0.406247,0.275637,[],[],3
3,22,40,F1,-22.0,0.277778,1,0.710264,0.286965,0.642788,[],[],4
4,39,30,F3,-39.0,0.333333,1,0.673028,0.545007,0.500000,[],[],5
...,...,...,...,...,...,...,...,...,...,...,...,...
59,-126,-2,P8,126.0,0.511111,1,-0.587427,-0.808524,-0.034899,[],[],60
60,-130,-25,P10,130.0,0.638889,1,-0.582563,-0.694272,-0.422618,[],[],61
61,-144,-2,PO8,144.0,0.511111,1,-0.808524,-0.587427,-0.034899,[],[],62
62,-155,16,PO4,155.0,0.411111,1,-0.871199,-0.406247,0.275637,[],[],63


chaninfo


,size,preview
sub-field,,
filename,1,/Users/AndrewGoldman/Desktop/Columbia Document...
filecontent,64,['1\t18\t-2\tFp1 ' '2\t36\t-2\tAF7 ' '3\...
plotrad,0,[]
shrink,0,[]
nosedir,1,+X
nodatchans,0,[]
icachansind,0,[]


event


... (5 outputs left out)

stats


,size,preview
sub-field,,
jp,0,[]
jpE,0,[]
icajp,0,[]
icajpE,0,[]
kurt,0,[]
kurtE,0,[]
icakurt,0,[]
icakurtE,0,[]
compenta,0,[]


etc


,size,preview
sub-field,,
T0,6,[2017 2 7 20 29 27]


### Check each participant's .fdt file

The `.fdt` file holds the EEG itself: the voltage (in µV) at every channel at every time point, stored as one long list of 32-bit numbers with no labels. They are in time order, all 64 channels for the first time point, then all 64 for the next, and so on, so the list can be cut into a table with one row per time point and one column per channel. The recording has 2048 time points per second.

This cell makes one row per participant showing:
- how many time points the file holds, and what percentage that is of the expected number (`RecordingDuration` × `SamplingFrequency` from `eeg.json`)
- whether the file stops part-way through a time point, a sign it was cut off
- how many values are missing or invalid (NaN or infinite)
- channels that never change (flat)
- channels stuck at the Biosemi recording limit (±262,144 µV) for most of the recording, which usually means the electrode wasn't working
- the typical size of the channels' ups and downs (median standard deviation, in µV)

The flat, stuck and size checks use one time point per second, which is enough to see each channel's overall behaviour. Reading all 40 files takes about a minute.

In [ ]:
# Read every participant's .fdt file (the EEG voltages, in µV)
rows = []
for f in sorted(root.glob("sub-*/eeg/*_eeg.fdt")):
    participant = f.name.split("_")[0]
    names = np.array(channels[participant]["name"])
    rate = eeg_json_report.loc[participant, "SamplingFrequency"]
    expected = eeg_json_report.loc[participant, "RecordingDuration"] * rate

    # One long list of 32-bit numbers (memmap reads it from disk as needed instead of all at once)
    values = np.memmap(f, dtype="<f4", mode="r")
    # Cut it into a table: one row per time point, one column per channel
    n_samples = values.size // len(names)
    data = values[: n_samples * len(names)].reshape(n_samples, len(names))
    # One time point per second, enough to see each channel's overall behaviour
    per_second = np.asarray(data[::rate])

    rows.append({
        "participant": participant,
        "n_samples": n_samples,
        "pct_of_expected": round(100 * n_samples / expected, 1),
        # Leftover values after the last full time point: the file stops part-way through one
        "ends_mid_sample": values.size % len(names) != 0,
        "invalid_values": int((~np.isfinite(data)).sum()),
        # Channel names, or "none" if there aren't any
        "flat_channels": ", ".join(names[per_second.std(axis=0) == 0]) or "none",
        # Near the ±262,144 µV limit for more than half the recording
        "channels_stuck_at_limit": ", ".join(names[np.median(np.abs(per_second), axis=0) > 250000]) or "none",
        "median_channel_sd_uV": round(float(np.median(per_second.std(axis=0)))),
    })
fdt_report = pd.DataFrame(rows).set_index("participant")

# Participants whose file holds less than the full recording
print("Incomplete recordings:", list(fdt_report.index[fdt_report["pct_of_expected"] < 100]))

# Checks where every participant has the same result, shown once
same_for_all = fdt_report.nunique() == 1
display(fdt_report.loc[:, same_for_all].iloc[0].to_frame("all participants"))

# Checks where participants differ, shown for every participant
display(fdt_report.loc[:, ~same_for_all])

### Summary

#### What we found

**Participants**
- 40 participants: 16 classical musicians (`Cl`) and 24 jazz-trained improvisers (`Imp`), aged 18–42.
- Gender is very unbalanced between groups: `Cl` is 12 women and 4 men, `Imp` is 4 women and 20 men.

**Files**
- Every participant has the same 8 files. `channels.tsv`, `electrodes.tsv`, `coordsystem.json` and `events.json` are identical for everyone; `eeg.json` differs only in the recording length.
- `channels.tsv`: all 64 channels are listed, but `type` and `units` are all `n/a` and there is no `status` (good/bad) column, so the dataset doesn't say which channels are bad.
- `coordsystem.json` and `electrodes.tsv`: the coordinate system and electrode positions are the same for every participant. The positions are a standard template (Biosemi's `biosemi64.sph`, textbook positions on a sphere), not measured on each person.
- Electrode names and channel names are the same, in the same order, in `channels.tsv`, `electrodes.tsv` and the `.set` file.
- `events.tsv`: every participant heard the same 1,440 trials (1,220 standard, 109 exemplar deviant, 111 function deviant), split into 8 blocks. Events are instantaneous markers (duration 0), except the 7 `boundary` events between blocks, which have no duration.
- `eeg.json` and `.set`: 64 EEG channels recorded at 2048 samples per second, with no cleaning done yet (raw data).
- `.fdt`: no missing or invalid values and no flat channels, but 12 recordings are incomplete and many have a channel that isn't working (see below).

#### Odd points that could cause confusion
- **Incomplete recordings.** 10 participants' EEG files hold only 1–8% of their recording (sub-003, 004, 005, 007, 010, 013, 019, 022, 025, 026), and 6 of their `.set` files can't be opened at all. sub-001 and sub-030 hold about 92%. This is how the files were published, not a download problem, and the checks that only ask "is the file there?" (including NEMAR's own report) don't catch it. Their `events.tsv` and `eeg.json` still describe the full session.
- **Stuck channels.** Fp1 is stuck at the recording limit (about 259,000 µV) for the whole recording in 26 participants, Fpz in 8, and P2 or P3 in a few others. These channels hold no usable EEG.
- **Values look far too large.** Channels vary by hundreds to thousands of µV instead of the usual tens, because raw Biosemi data has large offsets and slow drift until it is re-referenced and filtered. This is expected, not an error.
- **Labels that are wrong or misleading:**
  - `coordsystem.json` says positions are in mm, but they lie on a sphere of radius 1.
  - `eeg.json` says `MiscChannelCount: 64`, but the files hold only the 64 EEG channels.
  - `events.json` is an empty list (`[]`), so the event codes aren't documented in the dataset.
  - Every event's `trial_type` is `STATUS`, the name of the trigger channel; the real information is in `value`.
  - Channel `Afz` should be spelled `AFz` (EEG software matches names exactly).
- **README trial counts.** The README says 218 exemplar and 222 function deviants per participant, but every file has 109 and 111, which matches the README's own "7.5% each".
- **Event codes include responses.** `Correct - Function!` and similar codes, and the `correct`/`incorrect` endings on middle-chord codes, are scored button presses. They were added after recording (the original list in the `.set` file has a plain `16224` code for each press).
- **`16128` block markers are unreliable** (missing for sub-024 and once for sub-040); the `boundary` events mark the blocks reliably.

#### What this means for cleaning and pre-processing
1. **Decide who to include.** The 10 heavily truncated recordings can't be used. Keeping only the 28 complete ones leaves 11 `Cl` (8 women, 3 men) and 17 `Imp` (3 women, 14 men); including sub-001 and sub-030 gives 12 and 18. Group and gender are almost the same split, so any group difference could also be a gender difference.
2. **Fix the channel information.** Rename `Afz` to `AFz` and use the template positions, treating them as approximate.
3. **Mark bad channels** (Fp1, Fpz, P2, P3 where stuck) so they're left out of re-referencing and later filled in from neighbouring channels (interpolation). There are no separate eye (EOG) channels and Fp1/Fpz are the channels nearest the eyes, so eye blinks will need another method, such as ICA.
4. **Re-reference** (for example to the average of the good channels) to remove the large offsets.
5. **Filter**: a high-pass filter to remove slow drift, and removal of the 60 Hz mains noise (the recording was made in the US). The 8 blocks were joined together, so filtering should respect the `boundary` events to avoid artifacts at the joins.
6. **Downsample** from 2048 samples per second to a lower rate to make the data smaller and faster to work with.
7. **Cut the data into trials (epochs)** around the middle chord using the event codes. Deviants are only 15% of trials (7.5% each type), so later analysis needs to account for that imbalance. Every participant heard the trials in the same order within each block type, and recording date lines up with group (the later sessions were mostly improvisers), which matters if a model is trained to tell groups or trial types apart.